## Notebook — Camada Silver



In [0]:
catalog = "cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

#### 1) silver.tb_info_filmes (origem: tb_movies_info)

Mapeamento de colunas:
- Coluna Origem -> Coluna Destino
- id -> id_filme
- title -> titulo
- original_title -> titulo_original
- release_date -> data_lancamento
- runtime -> duracao_minutos
- original_language -> idioma_original
- status -> status_filme
- overview -> sinopse
- tagline -> frase_divulgacao'


In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# Verificar os diferentes valores de status
display(df_bronze_info.groupBy("status").count().orderBy(F.desc("count")))

**Limpeza e Tradução do Status**: Antes de traduzir o status_filme (Released → Lançado; Post Production →
Pós-Produção; In Production → Em Produção; Planned → Planejado; Rumored → Rumores; Canceled →
Cancelado), você deve normalizar a coluna. A coluna de status deve ser normalizada (removendo ruídos, hífens
sobressalentes e padronizando a caixa) antes da tradução dos termos para o português. Registros corrompidos ou
não mapeáveis devem ser padronizados como "Não Informado"

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

df_silver_info = (
    df_bronze_info
    .withColumn("status", F.trim(F.regexp_replace(F.lower(F.col("status")), r"[^a-z]+", " ")))
    .withColumn("status",
        F.when(F.col("status") == "released", "Lançado")
        .when(F.col("status") == "post production", "Pós-Produção")
        .when(F.col("status") == "in production", "Em Produção")
        .when(F.col("status") == "planned", "Planejado")
        .when(F.col("status") == "rumored", "Rumores")
        .when(F.col("status").isin("canceled", "cancelled"), "Cancelado")
        .otherwise("Não Informado")
    )
)

display(df_silver_info.groupBy("status").count().orderBy(F.desc("count")))

**Deduplicação**: A tabela deve conter unicidade por filme. Havendo registros duplicados na origem, mantenha
exclusivamente a versão mais recente com base na data de ingestão.


In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

print("linhas:", df_bronze_info.count())
print("ids distintos:", df_bronze_info.select("id").distinct().count())

In [0]:
from pyspark.sql import Window

w_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_silver_info = (
    df_silver_info
    .withColumn("id", F.trim(F.col("id").cast("string")))
    .filter(F.col("id").isNotNull() & (F.col("id") != ""))
    .withColumn("rn", F.row_number().over(w_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

print("linhas bronze:", df_bronze_info.count())
print("linhas silver:", df_silver_info.count())
print("ids distintos silver:", df_silver_info.select("id").distinct().count())

**Tratamento de Data Multi-Formato**: Converta o campo de data de lançamento testando os diferentes padrões
presentes na origem de forma robusta. Apenas valores onde a conversão for estritamente impossível devem ser
tratados como ausentes (NULL).

In [0]:
formatos_data = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy", "dd-MM-yyyy", "MM/dd/yyyy", "MM-dd-yyyy"]

df_silver_info = df_silver_info.withColumn(
    "release_date",
    F.coalesce(*[F.expr(f"try_to_date(trim(release_date), '{f}')") for f in formatos_data])
)

display(df_silver_info.agg(
    F.count("*").alias("total"),
    F.sum(F.col("release_date").isNull().cast("int")).alias("datas_nulas"),
))

In [0]:
# Depende da data já convertida (célula 4)
df_silver_info = df_silver_info.withColumn("ano_lancamento", F.year("release_date"))

display(df_silver_info.groupBy("ano_lancamento").count().orderBy("ano_lancamento"))

Mapeamento das colunas

In [0]:
df_silver_info = (
    df_silver_info
    .withColumn("runtime", F.expr("try_cast(try_cast(runtime AS double) AS int)"))
    .select("id", "title", "original_title", "release_date", "runtime",
            "original_language", "status", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

df_silver_info.printSchema()

In [0]:
df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

display(df_bronze_info.limit(10))
display(df_silver_info.limit(10))

%md
#### 2) silver.tb_financeiro_filmes (origem: tb_movies_financials)
Coluna Origem -> Coluna Destino
id -> id_filme
budget -> orcamento_usd
revenue -> receita_usd
orcamento_usd -> orcamento_brl
receita_usd -> receita_brl


Tratar valores textuais que representam ausência de dado (ex.: "Unknown", "Não Informado") como NULL antes
da conversão de tipo

In [0]:
df_bronze_fin = spark.table(f"{bronze_schema}.tb_movies_financials")
df_silver_fin = df_bronze_fin

textos_ausencia = ["unknown", "não informado", "nao informado", "n/a", "na", "null", "none", "nan", ""]

df_silver_fin = (
    df_silver_fin
    .withColumn("budget",
        F.when(F.lower(F.trim(F.col("budget").cast("string"))).isin(textos_ausencia), None)
        .otherwise(F.col("budget").cast("string"))
    )
    .withColumn("revenue",
        F.when(F.lower(F.trim(F.col("revenue").cast("string"))).isin(textos_ausencia), None)
        .otherwise(F.col("revenue").cast("string"))
    )
)

display(df_silver_fin.limit(10))

Higienize as colunas de orçamento e receita para remover símbolos de moedas, pontuações de milhar e textos representativos de ausência de dados. 

In [0]:
# Higienização direta de budget e revenue
# 1) Descobre o multiplicador pelo sufixo (M, K, B) ANTES de limpar, senão a letra é apagada
# 2) Remove tudo que não for dígito, ponto ou sinal de menos ($, R$, vírgulas de milhar, espaços, letras)
# 3) Converte com try_cast já aplicando o decimal e aplica o multiplicador
df_silver_fin = (
    df_silver_fin
    .withColumn("budget_mult",
        F.when(F.upper(F.trim(F.col("budget"))).rlike(r"[0-9]\s*M$"), 1000000)
        .when(F.upper(F.trim(F.col("budget"))).rlike(r"[0-9]\s*K$"), 1000)
        .when(F.upper(F.trim(F.col("budget"))).rlike(r"[0-9]\s*B$"), 1000000000)
        .otherwise(1)
    )
    .withColumn("revenue_mult",
        F.when(F.upper(F.trim(F.col("revenue"))).rlike(r"[0-9]\s*M$"), 1000000)
        .when(F.upper(F.trim(F.col("revenue"))).rlike(r"[0-9]\s*K$"), 1000)
        .when(F.upper(F.trim(F.col("revenue"))).rlike(r"[0-9]\s*B$"), 1000000000)
        .otherwise(1)
    )
    .withColumn("budget", F.regexp_replace(F.col("budget"), r"[^0-9.\-]", ""))
    .withColumn("revenue", F.regexp_replace(F.col("revenue"), r"[^0-9.\-]", ""))
    .withColumn("budget", F.expr("try_cast(budget AS decimal(38,2))") * F.col("budget_mult"))
    .withColumn("revenue", F.expr("try_cast(revenue AS decimal(38,2))") * F.col("revenue_mult"))
    .drop("budget_mult", "revenue_mult")
)

display(df_silver_fin)

Garanta que valores zerados ou negativos sejam tratados como ausentes.

In [0]:
df_silver_fin = (
    df_silver_fin
    .withColumn("budget", F.when(F.col("budget") > 0, F.col("budget")))
    .withColumn("revenue", F.when(F.col("revenue") > 0, F.col("revenue")))
)

df_silver_fin.printSchema()

In [0]:
# Conferir os valores
display(df_silver_fin.agg(
    F.count("*").alias("total"),
    F.sum(F.col("budget").isNull().cast("int")).alias("budget_nulo"),
    F.sum(F.col("revenue").isNull().cast("int")).alias("revenue_nulo"),
    F.min("budget").alias("budget_min"),
    F.min("revenue").alias("revenue_min"),
))

In [0]:
# Regra de negócio: a API foi consultada só para os últimos 7 dias, então não há cotação histórica por data de lançamento. Aplica-se a cotação mais recente disponível a todos os filmes.
taxa_dolar = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)
print("taxa aplicada (USD -> BRL):", taxa_dolar)

df_silver_fin = (
    df_silver_fin
    .withColumn("orcamento_brl", F.round(F.col("budget") * F.lit(float(taxa_dolar)), 2).cast("decimal(18,2)"))
    .withColumn("receita_brl", F.round(F.col("revenue") * F.lit(float(taxa_dolar)), 2).cast("decimal(18,2)"))
)

display(df_silver_fin.select("id", "budget", "orcamento_brl", "revenue", "receita_brl")
        .filter(F.col("budget").isNotNull()).limit(5))

Derive as colunas de Lucro (Dólar/Real) e Margem de Lucro Percentual, garantindo que operações aritméticas
com valores ausentes não invalidem o resultado e evitando divisões por zero.

In [0]:
# Regra de negócio: lucro = receita - orçamento. Se qualquer um dos dois for ausente (NULL), o lucro fica NULL em vez de ser calculado com 0, para não inventar um valor.
# Margem de lucro percentual = lucro / receita * 100. O when protege contra divisão por zero (receita zerada ou ausente -> NULL).
df_silver_fin = (
    df_silver_fin
    .withColumn("lucro_usd", (F.col("revenue") - F.col("budget")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    .withColumn("margem_lucro_percentual",
        F.when(F.col("revenue") > 0, F.round(F.col("lucro_usd") / F.col("revenue") * 100, 2))
    )
)

display(df_silver_fin.select("id", "budget", "revenue", "lucro_usd","orcamento_brl", "receita_brl", "lucro_brl", "margem_lucro_percentual").limit(10))

Deduplicação

In [0]:
from pyspark.sql import Window

w_dedup_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

print("linhas antes:", df_silver_fin.count())

df_silver_fin = (
    df_silver_fin
    .filter(F.col("id").isNotNull())
    .withColumn("rn", F.row_number().over(w_dedup_fin))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

print("linhas depois:", df_silver_fin.count())
print("ids distintos:", df_silver_fin.select("id").distinct().count())

In [0]:
df_silver_fin = (
    df_silver_fin
    .select("id", "budget", "revenue", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_percentual")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
)

df_silver_fin.printSchema()

In [0]:
df_silver_fin.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

display(spark.table(f"{silver_schema}.tb_financeiro_filmes").limit(10))

#### 3) silver.tb_metricas_engajamento (origem: tb_movies_metrics)
- Coluna Origem -> Coluna Destino
- id -> id_filme
- popularity -> popularidade
- vote_average -> nota_media_tmdb
- vote_count -> qtd_votos_tmdb
- averageRating -> nota_media_imdb
- numVotes -> qtd_votos_imdb

A coluna de popularidade apresenta inconsistências de pontuação e separadores decimais na origem. Limpe a
formatação numérica da coluna antes da conversão de tipo, garantindo que os valores não sejam invalidados ou
convertidos silenciosamente para nulos

In [0]:
df_bronze_met = spark.table(f"{bronze_schema}.tb_movies_metrics")
df_silver_met = df_bronze_met

df_bronze_met.printSchema()
print("linhas:", df_bronze_met.count())
print("ids distintos:", df_bronze_met.select("id").distinct().count())

# Verificar quais tipos de dados estão gravados no 'popularity'
display(
    df_bronze_met
    .filter(F.col("popularity").isNotNull() & F.expr("try_cast(popularity AS double)").isNull())
    .withColumn("padrao", F.regexp_replace(F.col("popularity").cast("string"), "[0-9]+", "9"))
    .groupBy("padrao").count().orderBy(F.desc("count"))
    .limit(30)
)


In [0]:
# Limpeza da formatação numérica da popularity, ANTES da conversão de tipo.
# Troca a vírgula por ponto só quando o texto é um número com vírgula, e tira espaços das pontas.
# Texto fora de contexto (column shift) não é alterado aqui; vira NULL na conversão.
df_silver_met = (
    df_silver_met
    .withColumn("popularity", F.trim(F.col("popularity")))
    .withColumn("popularity",
        F.when(F.col("popularity").rlike(r"^-?\d+,\d+$"), F.regexp_replace(F.col("popularity"), ",", "."))
        .otherwise(F.col("popularity"))
    )
)

# Conversão
# O que sobrou de texto fora de contexto vira NULL aqui.
df_silver_met = df_silver_met.withColumn("popularity", F.expr("try_cast(popularity AS double)"))

df_silver_met.printSchema()

display(df_silver_met.agg(
    F.count("*").alias("total"),
    F.sum(F.col("popularity").isNull().cast("int")).alias("popularidade_nula"),
    F.min("popularity").alias("minimo"),
    F.max("popularity").alias("maximo"),
))

Devido ao deslocamento de colunas (Column Shift) presente na base bruta, existem textos e atributos fora de
contexto espalhados pelas colunas de notas e contagens de votos. Aplique uma conversão de tipagem segura (para
valores decimais e inteiros), garantindo que textos ou caracteres incompatíveis sejam tratados como dados ausentes
(NULL) sem interromper a execução do pipeline.


In [0]:
df_silver_met = (
    df_silver_met
    .withColumn("vote_average", F.expr("try_cast(trim(vote_average) AS double)"))
    .withColumn("averageRating", F.expr("try_cast(trim(averageRating) AS double)"))
    .withColumn("vote_count", F.expr("try_cast(try_cast(trim(vote_count) AS double) AS int)"))
    .withColumn("numVotes", F.expr("try_cast(try_cast(trim(numVotes) AS double) AS int)"))
)

df_silver_met.printSchema()

display(df_silver_met.agg(
    F.count("*").alias("total"),
    F.count("vote_average").alias("tmdb_nota_ok"),
    F.count("averageRating").alias("imdb_nota_ok"),
    F.count("vote_count").alias("tmdb_votos_ok"),
    F.count("numVotes").alias("imdb_votos_ok"),
))

Identifique e trate inconsistências de escala e limites numéricos de negócio: Notas médias (TMDB e IMDb) que
estejam fora do intervalo válido de 0 a 10 (incluindo valores multiplicados por erro de escala) devem ser
desconsideradas e tratadas como NULL. Contagens de votos ou índices de popularidade com valores negativos
devem ser invalidados e tratados como NULL


In [0]:
df_silver_met = (
    df_silver_met
    .withColumn("vote_average", F.when(F.col("vote_average").between(0, 10), F.col("vote_average")))
    .withColumn("averageRating", F.when(F.col("averageRating").between(0, 10), F.col("averageRating")))
    .withColumn("vote_count", F.when(F.col("vote_count") >= 0, F.col("vote_count")))
    .withColumn("numVotes", F.when(F.col("numVotes") >= 0, F.col("numVotes")))
    .withColumn("popularity", F.when(F.col("popularity") >= 0, F.col("popularity")))
)

print("TMDB fora de 0-10 antes da regra:",
      df_bronze_met.filter(F.expr("try_cast(vote_average AS double)").isNotNull() &
                           ~F.expr("try_cast(vote_average AS double)").between(0, 10)).count())
print("IMDb fora de 0-10 antes da regra:",
      df_bronze_met.filter(F.expr("try_cast(averageRating AS double)").isNotNull() &
                           ~F.expr("try_cast(averageRating AS double)").between(0, 10)).count())

Deduplicação

In [0]:
from pyspark.sql import Window

colunas_metricas = ["popularity", "vote_average", "vote_count", "averageRating", "numVotes"]
completude = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in colunas_metricas)

w_dedup_met = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc(), completude.desc())

print("linhas antes:", df_silver_met.count())

df_silver_met = (
    df_silver_met
    .filter(F.col("id").isNotNull())
    .withColumn("rn", F.row_number().over(w_dedup_met))
    .filter(F.col("rn") == 1)
    .drop("rn", "ingestion_datetime")
)

print("linhas depois:", df_silver_met.count())
print("ids distintos:", df_silver_met.select("id").distinct().count())

In [0]:
df_silver_met = (
    df_silver_met
    .select("id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
)

df_silver_met.printSchema()

df_silver_met.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(spark.table(f"{silver_schema}.tb_metricas_engajamento").limit(10))

#### 4)

In [0]:
df_bronze_rev = spark.table(f"{bronze_schema}.tb_movies_reviews")
df_silver_rev = df_bronze_rev

df_bronze_rev.printSchema()
print("linhas:", df_bronze_rev.count())
print("linhas distintas (id, nome, nota, comentario):",
      df_bronze_rev.select("id", "nome", "nota", "comentario").distinct().count())

# Comentários nulos, vazios ou só com espaços
print("comentários nulos:", df_bronze_rev.filter(F.col("comentario").isNull()).count())
print("comentários vazios ou só espaços:",
      df_bronze_rev.filter(F.trim(F.col("comentario")) == "").count())

Remova registros integralmente duplicados (onde a combinação de filme, usuário, nota e comentário seja
idêntica), garantindo a unicidade das avaliações.


In [0]:
print("linhas antes:", df_silver_rev.count())

df_silver_rev = (
    df_silver_rev
    .dropDuplicates(["id", "nome", "nota", "comentario"])
)

print("linhas depois:", df_silver_rev.count())

Garanta que a nota atribuída pelo usuário respeite a regra de negócio da escala permitida (0 a 10). Qualquer valor
fora dessa faixa deve ser descartado e convertido para NULL. 

In [0]:
print("notas fora de 0-10 antes da regra:",
      df_silver_rev.filter(F.col("nota").isNotNull() & ~F.col("nota").between(0, 10)).count())

df_silver_rev = df_silver_rev.withColumn(
    "nota", F.when(F.col("nota").between(0, 10), F.col("nota"))
)

display(df_silver_rev.agg(F.min("nota").alias("nota_min"), F.max("nota").alias("nota_max")))

Identifique comentários não preenchidos ou compostos apenas por espaços em branco e preencha-os de forma
explícita com o texto padronizado "Sem comentário".

In [0]:
df_silver_rev = df_silver_rev.withColumn(
    "comentario",
    F.when(F.col("comentario").isNull() | F.col("comentario").rlike(r"^\s*$"), "Sem comentário")
     .otherwise(F.col("comentario"))
)

print("sem comentário:", df_silver_rev.filter(F.col("comentario") == "Sem comentário").count())
print("comentários nulos restantes:", df_silver_rev.filter(F.col("comentario").isNull()).count())

In [0]:
df_silver_rev = (
    df_silver_rev
    .select("id", "nome", "nota", "comentario")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
    # id_filme como string, igual às outras tabelas Silver
    .withColumn("id_filme", F.col("id_filme").cast("string"))
)

df_silver_rev.printSchema()

df_silver_rev.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(spark.table(f"{silver_schema}.tb_avaliacoes_usuarios").limit(10))

####5


In [0]:
df_bronze_cred = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_bronze_cred.printSchema()
print("linhas:", df_bronze_cred.count())
print("ids distintos:", df_bronze_cred.select("id").distinct().count())

# Exemplos de como o genres é escrito
display(df_bronze_cred.select("id", "genres").limit(20))

# Quais separadores aparecem
print("com vírgula:", df_bronze_cred.filter(F.col("genres").contains(",")).count())
print("com ponto e vírgula:", df_bronze_cred.filter(F.col("genres").contains(";")).count())
print("com barra:", df_bronze_cred.filter(F.col("genres").contains("|")).count())
print("genres nulo:", df_bronze_cred.filter(F.col("genres").isNull()).count())

In [0]:
# Remove letras e espaços; o que sobra são os caracteres "estranhos" (separadores, dígitos, pontuação)
# Depois separa em caracteres individuais e conta quantas vezes cada um aparece
display(
    df_bronze_cred
    .filter(F.col("genres").isNotNull())
    .withColumn("sobra", F.regexp_replace(F.col("genres"), r"[A-Za-z ]", ""))
    .withColumn("caractere", F.explode(F.split(F.col("sobra"), "")))
    .filter(F.col("caractere") != "")
    .groupBy("caractere").count().orderBy(F.desc("count"))
)

In [0]:
df_silver_gen = (
    df_bronze_cred
    .filter(F.col("id").isNotNull())
    .withColumn("genres", F.regexp_replace(F.col("genres"), r"[;|]", ","))
    .withColumn("genero", F.explode(F.split(F.col("genres"), ",")))
    .withColumn("genero", F.trim(F.col("genero")))
    .select("id", "genero")
)

In [0]:
# Resíduo de formatação: alguns gêneros vêm com aspas sobrando ("Horror\"", "Crime\""), resto do
# fechamento de campo no CSV. Remove as aspas para não perder gêneros legítimos.
df_silver_gen = (
    df_silver_gen
    .withColumn("genero", F.trim(F.regexp_replace(F.col("genero"), '"', '')))
    .dropDuplicates(["id", "genero"])
)

print("linhas:", df_silver_gen.count())
print("gêneros distintos:", df_silver_gen.select("genero").distinct().count())

display(
    df_silver_gen
    .groupBy("genero").count().orderBy(F.desc("count"))
    .limit(60)
)

In [0]:
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

print("linhas antes do filtro:", df_silver_gen.count())

df_descartados = df_silver_gen.filter(~F.col("genero").isin(generos_validos))
df_silver_gen = df_silver_gen.filter(F.col("genero").isin(generos_validos))

print("linhas depois do filtro:", df_silver_gen.count())
print("gêneros distintos:", df_silver_gen.select("genero").distinct().count())

# Removidos
display(
    df_descartados
    .groupBy("genero").count().orderBy(F.desc("count"))
    .limit(40)
)

display(df_silver_gen.groupBy("genero").count().orderBy(F.desc("count")))

In [0]:
df_silver_gen = (
    df_silver_gen
    .select("id", "genero")
    .withColumnRenamed("id", "id_filme")
    # id_filme como string, igual às outras tabelas Silver
    .withColumn("id_filme", F.col("id_filme").cast("string"))
)

df_silver_gen.printSchema()

df_silver_gen.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

display(spark.table(f"{silver_schema}.tb_generos").limit(10))

#### 6) silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers, production_companies)

Mapeamento de tipo de entidade:
- cast -> 'Ator'
- directors -> 'Diretor'
- writers -> 'Roteirista'
- production_companies -> 'Produtora'

In [0]:
# Verificar como as 4 colunas estão escritas (nulos, separadores e exemplos)
df_bronze_cred = spark.table(f"{bronze_schema}.tb_credits_and_tags")

for coluna in ["cast", "directors", "writers", "production_companies"]:
    print(f"=== {coluna} ===")
    print("nulos:", df_bronze_cred.filter(F.col(coluna).isNull()).count())
    print("com vírgula:", df_bronze_cred.filter(F.col(coluna).contains(",")).count())
    print("com ponto e vírgula:", df_bronze_cred.filter(F.col(coluna).contains(";")).count())
    print("com barra |:", df_bronze_cred.filter(F.col(coluna).contains("|")).count())
    display(df_bronze_cred.select("id", coluna).filter(F.col(coluna).isNotNull()).limit(8))

Dimensão unificada que consolida quatro tipos de entidade em uma única tabela (Manter o mapeamento de tipos
de entidade: cast -> ‘Ator’ etc.).

In [0]:
from functools import reduce

# Mapeamento: coluna de origem -> tipo_entidade
mapa_entidades = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

dfs_entidades = []
for coluna, tipo in mapa_entidades.items():
    df_tipo = (
        df_bronze_cred
        .filter(F.col("id").isNotNull() & F.col(coluna).isNotNull())
        # A Bronze repete cada filme várias vezes (várias cargas): remove cópias idênticas antes do explode
        .dropDuplicates(["id", coluna])
        # Padroniza os separadores ANTES do split (três aparecem na base: , ; |)
        .withColumn("nome", F.regexp_replace(F.col(coluna), r"[;|]", ","))
        .withColumn("nome", F.explode(F.split(F.col("nome"), ",")))
        .select(
            F.col("id").alias("id_filme"),
            F.col("nome").alias("nome_entidade"),
            F.lit(tipo).alias("tipo_entidade"),
        )
    )
    dfs_entidades.append(df_tipo)

# Une as quatro entidades em uma única tabela (dimensão unificada)
df_silver_pe = reduce(lambda a, b: a.unionByName(b), dfs_entidades)

print("linhas depois do explode e da união:", df_silver_pe.count())
display(df_silver_pe.groupBy("tipo_entidade").count().orderBy("tipo_entidade"))

 Padronize a formatação de capitalização de texto, consolide as entidades em um modelo unificado categorizado
pelo tipo de atuação (Ator, Diretor, Roteirista, Produtora) e elimine registros duplicados

In [0]:
# Regra de negócio: remove aspas sobrando e espaços repetidos antes de avaliar o que é resíduo
df_silver_pe = df_silver_pe.withColumn(
    "nome_entidade",
    F.trim(F.regexp_replace(F.regexp_replace(F.col("nome_entidade"), '"', ''), r"\s+", " "))
)

# Regra de negócio: o que NÃO é nome de pessoa ou empresa (resíduo do column shift)
eh_residuo = (
    (F.col("nome_entidade") == "")                                  # em branco
    | ~F.col("nome_entidade").rlike(r"\p{L}")                       # sem nenhuma letra (números deslocados)
    | F.col("nome_entidade").startswith("/")                        # caminho de arquivo
    | F.col("nome_entidade").rlike(r"(?i)\.(jpg|jpeg|png)$")        # nome de arquivo de imagem
    | F.col("nome_entidade").rlike(r"^[a-z]")                       # começa em minúscula: pedaço de frase
    | (F.size(F.split(F.col("nome_entidade"), " ")) > 8)            # texto longo demais para ser um nome
)

df_descartados_pe = df_silver_pe.filter(eh_residuo)
df_silver_pe = df_silver_pe.filter(~eh_residuo)

# Regra de negócio: padroniza a capitalização (Title Case) e depois elimina duplicatas.
# A deduplicação vem DEPOIS da capitalização, para "ROBERT DOWNEY JR." e "Robert Downey Jr." virarem o mesmo registro.
df_silver_pe = (
    df_silver_pe
    .withColumn("nome_entidade", F.initcap(F.lower(F.col("nome_entidade"))))
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

print("linhas finais:", df_silver_pe.count())
print("linhas descartadas como resíduo:", df_descartados_pe.count())

Conferência: o que foi descartado (deve ser só lixo) e uma amostra do que ficou.

In [0]:
# Descartados mais frequentes: devem ser só frases, números, caminhos de arquivo e vazios
display(
    df_descartados_pe
    .groupBy("tipo_entidade", "nome_entidade").count().orderBy(F.desc("count"))
    .limit(40)
)

# Entidades que ficaram, por tipo
display(df_silver_pe.groupBy("tipo_entidade").count().orderBy("tipo_entidade"))

# Nomes mais curtos e mais longos que ficaram (ajudam a achar resíduo que escapou)
display(df_silver_pe.withColumn("tamanho", F.length("nome_entidade")).orderBy("tamanho").limit(15))
display(df_silver_pe.withColumn("tamanho", F.length("nome_entidade")).orderBy(F.desc("tamanho")).limit(15))

Gravação da silver.tb_pessoas_empresas

In [0]:
df_silver_pe = (
    df_silver_pe
    .select("id_filme", "nome_entidade", "tipo_entidade")
    # id_filme como string, igual às outras tabelas Silver
    .withColumn("id_filme", F.col("id_filme").cast("string"))
)

df_silver_pe.printSchema()

df_silver_pe.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

display(spark.table(f"{silver_schema}.tb_pessoas_empresas").limit(10))

# Conferência final: nenhuma duplicata e só os 4 tipos de entidade
df_conf_pe = spark.table(f"{silver_schema}.tb_pessoas_empresas")
display(df_conf_pe.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("id_filme", "nome_entidade", "tipo_entidade").alias("registros_distintos"),
    F.countDistinct("tipo_entidade").alias("tipos"),
))

#### 7) silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)

In [0]:
df_bronze_cot = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_bronze_cot.printSchema()
print("linhas:", df_bronze_cot.count())
display(df_bronze_cot.orderBy("dataHoraCotacao").limit(20))

In [0]:
from pyspark.sql import Window

w_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc()
)

df_cot = (
    df_bronze_cot
    # data = primeiros 10 caracteres de dataHoraCotacao (yyyy-MM-dd); try_ não interrompe se o formato divergir
    .withColumn("data_cotacao", F.expr("try_to_date(substring(dataHoraCotacao, 1, 10), 'yyyy-MM-dd')"))
    .withColumn("cotacao_compra", F.expr("try_cast(cotacaoCompra AS decimal(10,4))"))
    .filter(F.col("data_cotacao").isNotNull() & F.col("cotacao_compra").isNotNull())
    .withColumn("rn", F.row_number().over(w_dia))
    .filter(F.col("rn") == 1)
    .select("data_cotacao", "cotacao_compra")
)

print("dias com cotação:", df_cot.count())
display(df_cot.orderBy("data_cotacao"))

In [0]:
# Calendário contínuo: um registro para cada dia entre a primeira e a última cotação
df_calendario = (
    df_cot.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim"))
    .selectExpr("explode(sequence(ini, fim, interval 1 day)) AS data_cotacao")
)

# Forward Fill: last(..., ignorenulls=True) pega o último valor não nulo até a linha atual.
# A janela ordena por data sem partição (a tabela tem poucas linhas, então não é problema de desempenho).
w_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cot = (
    df_calendario
    .join(df_cot, on="data_cotacao", how="left")
    # marca os dias que não tinham cotação (fins de semana e feriados) antes de preencher
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_ffill))
    .select("data_cotacao", "cotacao_compra", "cotacao_preenchida")
    .orderBy("data_cotacao")
)

display(df_silver_cot)

Gravação da silver.tb_cotacao_dolar e conferência da série contínua

In [0]:
df_silver_cot.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

df_conf_cot = spark.table(f"{silver_schema}.tb_cotacao_dolar")
df_conf_cot.printSchema()

# Conferência: dias esperados (fim - início + 1) == linhas; nenhuma cotação nula; dias preenchidos
display(df_conf_cot.agg(
    F.count("*").alias("linhas"),
    (F.datediff(F.max("data_cotacao"), F.min("data_cotacao")) + 1).alias("dias_esperados"),
    F.sum(F.col("cotacao_compra").isNull().cast("int")).alias("cotacoes_nulas"),
    F.sum(F.col("cotacao_preenchida").cast("int")).alias("dias_preenchidos"),
))